# Heston GPU Monte Carlo — Colab quick run

**Runtime → Change runtime type → GPU** (T4 / L4 / …).

If this notebook lives in a **subfolder** of a monorepo, set `PROJECT_SUBDIR` (see cell 2).

In [ ]:
!nvidia-smi

In [ ]:
import os, shutil, subprocess
from pathlib import Path

# --- edit for your GitHub repository ---
REPO_URL = "https://github.com/chezke/Monte-Carlo-simulation-of-Heston-model.git"
BRANCH = "main"
# If repo root is *not* this project, cd into it after clone (use "" if notebook lives at repo root):
PROJECT_SUBDIR = ""  # example: "Monte Carlo simulation of Heston model" or "heston-mc"

CLONE_DIR = "/content/heston_mc_src"
SAFE_DIR = "/content"

def safe_chdir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)
    os.chdir(path)
    print("cwd:", os.getcwd())
safe_chdir(SAFE_DIR)

if os.path.isdir(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)
subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, CLONE_DIR],
    check=True,
)
os.chdir(CLONE_DIR)
if PROJECT_SUBDIR:
    os.chdir(PROJECT_SUBDIR)
print("cwd:", os.getcwd())
!ls -la

In [ ]:
# Colab GPU images usually ship nvcc; if 'make' fails, try: which nvcc
!make clean 2>/dev/null; make NVFLAGS="-O3 -std=c++14 -Iinclude"

In [ ]:
!./bin/MC_Euler

In [ ]:
!./bin/MC_exact

In [ ]:
!./bin/MC_benchmark_Q3 > results_q3.csv

In [ ]:
# Q3 sweep can be slow; reduce grid/path count via NVFLAGS if needed
# !make NVFLAGS="-O3 -std=c++14 -Iinclude -DHESTON_Q3_N_PATHS=16384" bin/MC_benchmark_Q3
!./bin/MC_benchmark_Q3 | head -20

## Q3 Analysis — Euler vs Almost-exact scheme

We load the benchmark CSV and compare:
1. **Execution time** of each method
2. **Pricing bias** (error vs Broadie–Kaya exact reference)
3. **Impact of Δt = 1/30** on the almost-exact scheme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("results_q3.csv", comment="#")
print(f"Loaded {len(df)} parameter combinations (Feller-filtered)")
df.head()

### 1. Execution time comparison

The Euler scheme uses a simple arithmetic update per step, while the exact and almost-exact schemes require Poisson + Gamma sampling at each step. We expect Euler to be **much faster per run** for the same number of time steps.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Box plot of execution times ---
time_cols = {
    "Exact (BK)\nΔt=1/1000": "ms_exact",
    "Euler\nΔt=1/1000": "ms_euler",
    "Almost-exact\nΔt=1/1000": "ms_almost_dt1000",
    "Almost-exact\nΔt=1/30": "ms_almost_dt1_30",
}
time_data = [df[c].values for c in time_cols.values()]

bp = axes[0].boxplot(time_data, labels=time_cols.keys(), patch_artist=True)
colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel("Kernel time (ms)")
axes[0].set_title("Execution time distribution across parameter grid")
axes[0].set_yscale("log")
axes[0].grid(axis="y", alpha=0.3)

# --- Mean execution time bar chart ---
means = [df[c].mean() for c in time_cols.values()]
bars = axes[1].bar(time_cols.keys(), means, color=colors, alpha=0.7, edgecolor="black")
for bar, m in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, m + 2, f"{m:.1f}", ha="center", va="bottom", fontsize=10)
axes[1].set_ylabel("Mean kernel time (ms)")
axes[1].set_title("Average execution time")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("fig_q3_time.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
print("Mean execution time (ms):")
for name, col in time_cols.items():
    print(f"  {name:30s}  {df[col].mean():8.2f} ± {df[col].std():6.2f}")
print(f"\nSpeedup  Euler / Exact (BK):            {df['ms_exact'].mean() / df['ms_euler'].mean():.1f}×")
print(f"Speedup  Almost Δt=1/30 / Almost Δt=1/1000: {df['ms_almost_dt1000'].mean() / df['ms_almost_dt1_30'].mean():.1f}×")

### 2. Pricing bias comparison

`err_*` = `mean_* − mean_exact`, where the Broadie–Kaya exact scheme (dt = 1/1000) serves as the MC reference. Since all runs use the same number of paths (262 144), the difference includes both **discretization bias** and residual **MC noise**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

err_cols = {
    "Euler (Δt=1/1000)": "err_euler",
    "Almost-exact (Δt=1/1000)": "err_almost_1000",
    "Almost-exact (Δt=1/30)": "err_almost_30",
}
colors_err = ["#dd8452", "#55a868", "#c44e52"]

# --- (a) Histogram of absolute errors ---
for i, (label, col) in enumerate(err_cols.items()):
    axes[0].hist(df[col].abs().values, bins=30, alpha=0.55, label=label, color=colors_err[i], edgecolor="black", linewidth=0.4)
axes[0].set_xlabel("|bias + MC noise|")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of |err| across parameter grid")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# --- (b) |err| vs feller_gap ---
for i, (label, col) in enumerate(err_cols.items()):
    axes[1].scatter(df["feller_gap"], df[col].abs(), s=8, alpha=0.5, label=label, color=colors_err[i])
axes[1].set_xlabel("Feller gap  (2κθ − σ²)")
axes[1].set_ylabel("|err|")
axes[1].set_title("|Bias| vs Feller margin")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# --- (c) Box plot of absolute errors ---
abs_data = [df[c].abs().values for c in err_cols.values()]
bp2 = axes[2].boxplot(abs_data, labels=["Euler\nΔt=1/1000", "Almost\nΔt=1/1000", "Almost\nΔt=1/30"], patch_artist=True)
for patch, color in zip(bp2["boxes"], colors_err):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_ylabel("|err|")
axes[2].set_title("|Bias + MC noise| distribution")
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("fig_q3_bias.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary statistics
print("Absolute error statistics:")
print(f"  {'Method':<28s}  {'mean |err|':>10s}  {'median':>10s}  {'max':>10s}")
for label, col in err_cols.items():
    ae = df[col].abs()
    print(f"  {label:<28s}  {ae.mean():10.6f}  {ae.median():10.6f}  {ae.max():10.6f}")

### 3. Impact of Δt = 1/30 on the almost-exact scheme

The almost-exact scheme updates log $S$ with coefficients $k_0, k_1, k_2$ that are **linear in Δt**, while the variance $v$ is drawn from its **exact** CIR transition regardless of Δt. Reducing from 1000 to 30 time steps therefore:

- **Speeds up** the simulation dramatically (fewer Poisson + Gamma draws per path).
- **Increases discretization bias** in log $S$ because the trapezoidal-style approximation of $\int v\,ds$ and the linearisation of the drift become coarser.

Below we directly compare the two step sizes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- (a) Scatter: |err| Δt=1/1000 vs Δt=1/30 ---
axes[0].scatter(df["err_almost_1000"].abs(), df["err_almost_30"].abs(), s=10, alpha=0.5, c=df["feller_gap"], cmap="viridis")
lim = max(df["err_almost_1000"].abs().max(), df["err_almost_30"].abs().max()) * 1.05
axes[0].plot([0, lim], [0, lim], "k--", lw=0.8, label="y = x")
axes[0].set_xlabel("|err|  Almost-exact Δt = 1/1000")
axes[0].set_ylabel("|err|  Almost-exact Δt = 1/30")
axes[0].set_title("Bias: fine vs coarse Δt")
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- (b) Execution time Δt=1/1000 vs Δt=1/30 ---
axes[1].scatter(df["ms_almost_dt1000"], df["ms_almost_dt1_30"], s=10, alpha=0.5, color="#c44e52")
axes[1].set_xlabel("Time (ms) Δt = 1/1000")
axes[1].set_ylabel("Time (ms) Δt = 1/30")
axes[1].set_title("Execution time: fine vs coarse Δt")
axes[1].grid(alpha=0.3)

# --- (c) |err_30| vs sigma (volatility of vol) ---
axes[2].scatter(df["sigma"], df["err_almost_30"].abs(), s=10, alpha=0.5, c=df["kappa"], cmap="coolwarm")
axes[2].set_xlabel("σ (vol-of-vol)")
axes[2].set_ylabel("|err|  Almost-exact Δt = 1/30")
axes[2].set_title("Coarse-Δt bias grows with σ")
cb = plt.colorbar(axes[2].collections[0], ax=axes[2])
cb.set_label("κ")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_q3_dt_impact.png", dpi=150, bbox_inches="tight")
plt.show()

# Fraction where coarse Δt has larger error
worse = (df["err_almost_30"].abs() > df["err_almost_1000"].abs()).mean()
print(f"Fraction of cases where |err(Δt=1/30)| > |err(Δt=1/1000)|: {worse:.1%}")
print(f"Mean |err|  Δt=1/1000: {df['err_almost_1000'].abs().mean():.6f}")
print(f"Mean |err|  Δt=1/30  : {df['err_almost_30'].abs().mean():.6f}")
print(f"Ratio: {df['err_almost_30'].abs().mean() / df['err_almost_1000'].abs().mean():.2f}×")
print(f"\nMean time  Δt=1/1000: {df['ms_almost_dt1000'].mean():.1f} ms")
print(f"Mean time  Δt=1/30  : {df['ms_almost_dt1_30'].mean():.1f} ms")
print(f"Speedup: {df['ms_almost_dt1000'].mean() / df['ms_almost_dt1_30'].mean():.1f}×")

### 4. Summary and conclusions

**Execution time.**
The Euler scheme is by far the fastest (only simple arithmetic per step). The exact (BK) and almost-exact schemes at Δt = 1/1000 are comparably slow because both require Poisson + Gamma sampling at each of the 1000 steps. Reducing to Δt = 1/30 cuts the almost-exact time by roughly one order of magnitude — making it competitive with (or even faster than) the Euler scheme.

**Pricing accuracy.**
With Δt = 1/1000, the almost-exact scheme achieves the smallest bias because the variance is drawn from its **exact** CIR transition (no truncation artifacts like Euler) and the log-price linearisation error is negligible at small Δt. The Euler scheme shows larger bias, especially when the Feller margin is small (variance near zero, truncation kicks in).

**Impact of Δt = 1/30.**
Switching from 1000 to 30 steps increases the mean absolute error of the almost-exact scheme noticeably. The scatter plot confirms that in the majority of parameter combinations, |err(Δt = 1/30)| > |err(Δt = 1/1000)|. The degradation is worst for **large σ** (high vol-of-vol) and **small κ** (slow mean reversion), because the variance path is more volatile and the linear-in-Δt approximation of $\int v\,ds$ in the log-price update becomes less accurate.

**Trade-off.** Δt = 1/30 is a practical choice when a moderate increase in bias is acceptable in exchange for a large speedup. For high-precision pricing (small confidence intervals), Δt = 1/1000 or the full Broadie–Kaya exact scheme is preferable.